In [0]:
from pyspark.sql import functions as F

batch_id = "batch_001"

control_rows = (
    spark.table("workspace.migration_dev.batch_control")
    .filter(F.col("batch_id") == batch_id)
    .collect()
)

if len(control_rows) != 1:
    raise RuntimeError("Expected exactly one batch control record")

control = control_rows[0]

if control["batch_status"] != "APPROVED" or control["failed_checks"] != 0:
    raise RuntimeError("Batch is not approved; Gold publication blocked")

silver = (
    spark.table("workspace.migration_dev.silver_documents")
    .filter(F.col("batch_id") == batch_id)
)

# Silver must belong to the current approval run.
if silver.filter(
    ~F.col("validation_run_id").eqNullSafe(control["validation_run_id"])
).limit(1).count() > 0:
    raise RuntimeError("Silver approval is outdated; rerun the Silver notebook")

if silver.count() != control["expected_document_count"]:
    raise RuntimeError("Silver count differs from the approved batch")

if silver.groupBy("document_id").count().filter(
    F.col("count") > 1
).limit(1).count() > 0:
    raise RuntimeError("Duplicate document IDs; Gold publication blocked")

gold = silver.select(
    "batch_id",
    "document_id",
    "account_number",
    "document_type",
    "statement_date",
    "file_name",
    "pdf_path",
    "source_sha256",
    "file_size_bytes",
    "validation_run_id",
    F.current_timestamp().alias("published_at"),
)

gold.createOrReplaceTempView("current_gold_batch")

spark.sql("""
    MERGE INTO workspace.migration_dev.gold_load_manifest AS target
    USING current_gold_batch AS source
    ON target.batch_id = source.batch_id
       AND target.document_id = source.document_id
    WHEN MATCHED
         AND target.validation_run_id <> source.validation_run_id
         THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

saved = (
    spark.table("workspace.migration_dev.gold_load_manifest")
    .filter(F.col("batch_id") == batch_id)
)

print(f"Gold loading instructions: {saved.count()}")
display(saved.orderBy("document_id").limit(5))

Gold loading instructions: 100


batch_id,document_id,account_number,document_type,statement_date,file_name,pdf_path,source_sha256,file_size_bytes,validation_run_id,published_at
batch_001,DOC000001,TEST00000001,ACCOUNT_STATEMENT,2026-09-30,DOC000001.pdf,/Volumes/workspace/migration_dev/landing/batch_001/pdfs/DOC000001.pdf,12298c6adcbc1f0ffed046f3c14ab064da327b4fab519c9e25014e6a8182c431,1542,4e66d857-a136-4b69-af88-a9acada6bd83,2026-10-14T10:13:12.455Z
batch_001,DOC000002,TEST00000002,ACCOUNT_STATEMENT,2026-09-30,DOC000002.pdf,/Volumes/workspace/migration_dev/landing/batch_001/pdfs/DOC000002.pdf,e45b766bfe7a3f38c7303239ffab722e244474ccb300ef84529b48197cbb83ce,1542,4e66d857-a136-4b69-af88-a9acada6bd83,2026-10-14T10:13:12.455Z
batch_001,DOC000003,TEST00000003,ACCOUNT_STATEMENT,2026-09-30,DOC000003.pdf,/Volumes/workspace/migration_dev/landing/batch_001/pdfs/DOC000003.pdf,eecd9ffaf648015b1dcc0a49067b9724e9a38b609a1174a19046bf6d33a662c7,1543,4e66d857-a136-4b69-af88-a9acada6bd83,2026-10-14T10:13:12.455Z
batch_001,DOC000004,TEST00000004,ACCOUNT_STATEMENT,2026-09-30,DOC000004.pdf,/Volumes/workspace/migration_dev/landing/batch_001/pdfs/DOC000004.pdf,8a1b437face5c2d7feb3fbee111a1b98b3f72afe81110e92fa55279fc7e3ea76,1542,4e66d857-a136-4b69-af88-a9acada6bd83,2026-10-14T10:13:12.455Z
batch_001,DOC000005,TEST00000005,ACCOUNT_STATEMENT,2026-09-30,DOC000005.pdf,/Volumes/workspace/migration_dev/landing/batch_001/pdfs/DOC000005.pdf,e9907b9c7b5e0b6d0836ae782f5a5de81780eb22dbba4802102a97ed69773eb6,1542,4e66d857-a136-4b69-af88-a9acada6bd83,2026-10-14T10:13:12.455Z
